### Overview
In this notebook, we will be looking at sample outputs of SQLAgent from part 2. Main objective is querying very large databases without loading whole data

Challenges:
1. How do we query database with 500+ columns? - retrieval based approach where we find top k columns to answer the user query - gives around 99% accuracy based on my evaluation data - refer retrieve_performance.ipynb
2. How do we incorporate row information? - custom SQL prompt (current langchain is not allowing this change, so modified the langchain source code to incorporate custom sql prompts)
3. Writing valid sql queries? Prompt is changed such that both the writing and validation are in one query
4. Extracting SQL from noisy GPT outputs? Using regular expressions and string matching
5. How do we answer the queries? - Pandas agent can help further analyze the SQL output (all functionalities like tools, answering generic questions,  can be intergrated with sql agent also)

There are two types of queries we discuss in this demo
1. Point queries - ex: how many rows in the data? which has single output - deterministic
2. Complex queries - ex: average fire size for each fire size class - we show how the agent intermediate steps also to understand the agent behavior, can be modified accordingly


In [ ]:
import sys
sys.path.append(".././part2")
from sql_agent import SQLAgent

### Point Queries

In [ ]:
my_agent = SQLAgent(verbose=False)
print(my_agent.run(question = "Number of rows in the sql data"))

Retrieval Agent Configurations
Model :  gpt-3.5-turbo
k (no. of columns to extract):  5
Number of rows in the sql data : 1880465


In [ ]:
print(my_agent.run(question = "How many columns are in the data"))

How many columns are in the data : 39


In [ ]:
print(my_agent.run(question = "average of fire size in the data"))

average of fire size in the data : 74.520158339107


In [ ]:
print(my_agent.run(question = "number of null values in source system type"))

number of null values in source system type : 0


### Sample Queries
- no intermediate steps and with intermediate steps to understand how agent is answering the user query
- currently our SQL agent can fetch and then answer (next step for this agent can be answering generic queries - already pandas agent has this functionality, I need to integrate SQL agent with pandas agent to achieve this behavior)

In [ ]:
print(my_agent.run(question = "Average fire size for each fire size class"))

/project/pi_hongyu_umass_edu/zonghai/clinical-llm-alignment/durga_sandeep/Aira/AiraEnv/lib/python3.9/site-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:242: UserWarning: Received additional kwargs {'agent': <AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION: 'chat-zero-shot-react-description'>} which are no longer supported.
  warnings.warn(


The average fire size for each fire size class is:
A: 0.118801
B: 2.146998
C: 28.531914
D: 161.801034
E: 512.854904


In [ ]:
my_agent = SQLAgent(verbose=True) # verbose is True
print(my_agent.run(question = "Average fire size for each fire size class"))

Question: Average fire size for each fire size class
Top k Columns: ['FIRE_SIZE_CLASS', 'FIRE_SIZE', 'FIRE_YEAR', 'STAT_CAUSE_DESCR', 'STATE', 'SOURCE_REPORTING_UNIT_NAME', 'SOURCE_SYSTEM']


SQL Query Raw: 
First draft: SELECT "FIRE_SIZE_CLASS", AVG("FIRE_SIZE") AS avg_fire_size
FROM fires
GROUP BY "FIRE_SIZE_CLASS"
LIMIT 5;


SQL Final Query: 
SELECT "FIRE_SIZE_CLASS", AVG("FIRE_SIZE") AS avg_fire_size
FROM fires
GROUP BY "FIRE_SIZE_CLASS"
LIMIT 5;




/project/pi_hongyu_umass_edu/zonghai/clinical-llm-alignment/durga_sandeep/Aira/AiraEnv/lib/python3.9/site-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:242: UserWarning: Received additional kwargs {'agent': <AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION: 'chat-zero-shot-react-description'>} which are no longer supported.
  warnings.warn(


A: 0.118801
B: 2.146998
C: 28.531914
D: 161.801034
E: 512.854904
